### Problem (unicode1): Understanding Unicode (1 point)

1. What Unicode character does chr(0) return?

In [4]:
chr(0)

'\x00'

2. How does this character’s string representation (__repr__()) differ from its printed representa-
tion?

In [7]:
print(chr(0))

 


In [11]:
chr(0) == ""

False

In [12]:
len(chr(0))

1

In [8]:
chr(0).__repr__()

"'\\x00'"

3. What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:
    ```Python
    chr(0)
    print(chr(0))
    "this is a test" + chr(0) + "string"
    print("this is a test" + chr(0) + "string")
    ```

In [14]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [15]:
print("this is a test" + chr(0) + "string")

this is a test string


---

In [26]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'


In [21]:
print(type(utf8_encoded))

<class 'bytes'>


In [22]:
list(utf8_encoded)

[104,
 101,
 108,
 108,
 111,
 33,
 32,
 227,
 129,
 147,
 227,
 130,
 147,
 227,
 129,
 171,
 227,
 129,
 161,
 227,
 129,
 175,
 33]

In [23]:
print(len(test_string))
print(len(utf8_encoded))

13
23


In [25]:
print(utf8_encoded.decode("utf-8"))

hello! こんにちは!


### Problem (unicode2): Unicode Encodings (3 points)

1. What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.

In [18]:
text = "hello world"
print(f"Length of text: {len(text)}")
print(f"UTF-8 encoding: {text.encode('utf-8')}")
print(f"UTF-16 encoding: {text.encode('utf-16')}")
print(f"UTF-32 encoding: {text.encode('utf-32')}")
print(f"Length of UTF-8 encoding: {len(text.encode('utf-8'))}")
print(f"Length of UTF-16 encoding: {len(text.encode('utf-16'))}")
print(f"Length of UTF-32 encoding: {len(text.encode('utf-32'))}")

Length of text: 11
UTF-8 encoding: b'hello world'
UTF-16 encoding: b'\xff\xfeh\x00e\x00l\x00l\x00o\x00 \x00w\x00o\x00r\x00l\x00d\x00'
UTF-32 encoding: b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00 \x00\x00\x00w\x00\x00\x00o\x00\x00\x00r\x00\x00\x00l\x00\x00\x00d\x00\x00\x00'
Length of UTF-8 encoding: 11
Length of UTF-16 encoding: 24
Length of UTF-32 encoding: 48


2. Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.
    ```python
    def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
        return "".join([bytes([b]).decode("utf-8") for b in bytestring])
    >>> decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
    'hello'
    ```

In [19]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

In [22]:
decode_utf8_bytes_to_str_wrong("こんにちは".encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data

3. Give a two byte sequence that does not decode to any Unicode character(s).

In [42]:
# A continuation byte CANNOT appear at the start
b"\x80\x80".decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte

In [43]:
# 0x41 is 'A' (an ASCII byte), not a continuation byte
b"\xc2\x41".decode("utf-8")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc2 in position 0: invalid continuation byte

## 2.4 BPE Tokenizer Training

### Pre-tokenization

In [11]:
import regex as re

In [12]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [13]:
re.findall(PAT, "some text that i'll pre-tokenize")

['some', ' text', ' that', ' i', "'ll", ' pre', '-', 'tokenize']

### Example (bpe_example): BPE training example

In [39]:
from collections import Counter

In [1]:
text = """
low low low low low
lower lower widest widest widest
newest newest newest newest newest newest
"""

In [97]:
vocabulary_b2i = {}
num_bytes = 256
for i in range(num_bytes):
    vocabulary_b2i[bytes([i])] = len(vocabulary_b2i)

vocabulary_b2i[b"<|endoftext|>"] = len(vocabulary_b2i)

assert len(vocabulary_b2i) == num_bytes + 1

vocabulary_i2b = {v: k for k, v in vocabulary_b2i.items()}

In [95]:
def pre_tokenize(text: str) -> dict[tuple[bytes], int]:
    freq = Counter(text.split())
    results = {}
    for k, v in freq.items():
        key = tuple(bytes([b]) for b in k.encode("utf-8"))
        results[key] = v

    return results

In [157]:
pre_tokenes = pre_tokenize(text)
pre_tokenes

{(b'l', b'o', b'w'): 5,
 (b'l', b'o', b'w', b'e', b'r'): 2,
 (b'w', b'i', b'd', b'e', b's', b't'): 3,
 (b'n', b'e', b'w', b'e', b's', b't'): 6}

In [159]:
def get_most_freq_pair(pre_tokenes: dict[tuple[bytes], int]) -> tuple[bytes]:
    merges = {}
    for k, v in pre_tokenes.items():
        if len(k) == 1:
            continue

        for i in range(len(k)):
            if i + 1 >= len(k):
                continue
            pair = k[i : i + 2]
            if pair not in merges:
                merges[pair] = 0
            merges[pair] += v

    most_freq_pair = max(merges, key=lambda x: (merges[x], x))
    return most_freq_pair

In [160]:
most_freq_pair = get_most_freq_pair(pre_tokenes)
most_freq_pair

(b's', b't')

In [171]:
def merge_tokens(
    pre_tokenes: dict[tuple[bytes], int], most_freq_pair: tuple[bytes]
) -> dict[tuple[bytes], int]:
    new_pre_tokenes = {}
    merged = b"".join(most_freq_pair)
    a, b = most_freq_pair

    for k, v in pre_tokenes.items():
        new_k = []
        i = 0
        while i < len(k):
            if i + 1 < len(k) and k[i] == a and k[i + 1] == b:
                new_k.append(merged)
                i += 2
            else:
                new_k.append(k[i])
                i += 1
        new_pre_tokenes[tuple(new_k)] = v

    return new_pre_tokenes

In [172]:
new_pre_tokenes = merge_tokens(pre_tokenes, most_freq_pair)
new_pre_tokenes

{(b'l', b'o', b'w'): 5,
 (b'l', b'o', b'w', b'e', b'r'): 2,
 (b'w', b'i', b'd', b'e', b'st'): 3,
 (b'n', b'e', b'w', b'e', b'st'): 6}